# 2.a Benchmark — easy-search + 评估

对 `work/DB/` 中各方法库跑 **easy-search**（query = target），再按 scope_family 协议评估，写出灵敏度表与 `auc_easy.csv`（**绘图在 `3.plot.ipynb`**）。

- Foldseek / 预测方法：`bin/foldseek easy-search`
- MMseqs2：`bin/mmseqs easy-search`

共享配置：[`config.py`](config.py)。

> easy-search 较吃 CPU。登录节点可把 `THREADS` 调小。


In [ ]:
from __future__ import annotations

import gc
import os
import re
import subprocess
import traceback
from pathlib import Path

import pandas as pd

from config import (
    AA_FASTA,
    ALN_DIR,
    DBS_DIR,
    EASY_SEARCH_PARAMS,
    FOLDSEEK_BIN,
    METRICS_DIR,
    METHODS,
    MMSEQS_BIN,
    PROJECT_ROOT,
    SCOP_LOOKUP,
    WORK_DIR,
    aln_tmp_dir,
    aln_tsv,
    db_prefix,
    cleanup_tmp,
    ensure_work_dirs,
    method_engine,
    metric_prefix,
    require_project_root,
    scop_cla_path,
)

ROOT = require_project_root("2.a.benchmark.ipynb")
assert ROOT == PROJECT_ROOT

SKIP_EXISTING = True
THREADS = EASY_SEARCH_PARAMS["threads"]  # 可改为 8 / 16
ONLY_METHODS = None  # 例如 ["foldseek", "mmseqs", "ESM3_LoRA"]；None = 全部

ensure_work_dirs()
print("ROOT:", ROOT)
print("foldseek:", FOLDSEEK_BIN)
print("mmseqs:", MMSEQS_BIN)
print("DB:", DBS_DIR)
print("方法:", [(k, eng) for _, k, eng, _ in METHODS])
print("THREADS =", THREADS)


## Phase A — easy-search

输出：`work/aln/{method}_easy.tsv`

- `foldseek` / 预测方法 → Foldseek
- `mmseqs` → MMseqs2（`bin/mmseqs`）


In [ ]:
def foldseek_easy_search(
    query_db: Path,
    output_tsv: Path,
    tmp_dir: Path,
    target_db: Path | None = None,
    threads: int | None = None,
    skip_existing: bool = True,
) -> Path:
    target_db = Path(target_db or query_db)
    threads = int(threads if threads is not None else EASY_SEARCH_PARAMS["threads"])

    if skip_existing and output_tsv.is_file() and output_tsv.stat().st_size > 0:
        print(f"⏭️  比对已存在，跳过: {output_tsv} ({output_tsv.stat().st_size} bytes)")
        return output_tsv

    if not FOLDSEEK_BIN.is_file():
        raise FileNotFoundError(f"foldseek 不存在: {FOLDSEEK_BIN}")
    if not query_db.is_file():
        raise FileNotFoundError(f"query DB 不存在: {query_db}")
    if not target_db.is_file():
        raise FileNotFoundError(f"target DB 不存在: {target_db}")

    output_tsv.parent.mkdir(parents=True, exist_ok=True)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        str(FOLDSEEK_BIN), "easy-search",
        str(query_db), str(target_db), str(output_tsv), str(tmp_dir),
        "--threads", str(threads),
        "-s", str(EASY_SEARCH_PARAMS["sensitivity"]),
        "--max-seqs", str(EASY_SEARCH_PARAMS["max_seqs"]),
        "-e", str(EASY_SEARCH_PARAMS["evalue"]),
    ]
    print("[CMD]", " ".join(cmd))
    subprocess.run(cmd, check=True)
    if not output_tsv.is_file():
        raise FileNotFoundError(f"结果未生成: {output_tsv}")
    print(f"✅ {output_tsv}")
    return output_tsv


def mmseqs_easy_search(
    query_db: Path,
    output_tsv: Path,
    tmp_dir: Path,
    target_db: Path | None = None,
    threads: int | None = None,
    skip_existing: bool = True,
) -> Path:
    target_db = Path(target_db or query_db)
    threads = int(threads if threads is not None else EASY_SEARCH_PARAMS["threads"])

    if skip_existing and output_tsv.is_file() and output_tsv.stat().st_size > 0:
        print(f"⏭️  比对已存在，跳过: {output_tsv} ({output_tsv.stat().st_size} bytes)")
        return output_tsv

    if not MMSEQS_BIN.is_file():
        raise FileNotFoundError(
            f"mmseqs 不存在: {MMSEQS_BIN}\n请先运行 0.prepare.ipynb 下载 MMseqs"
        )
    if not query_db.is_file():
        raise FileNotFoundError(f"query DB 不存在: {query_db}")
    if not target_db.is_file():
        raise FileNotFoundError(f"target DB 不存在: {target_db}")

    output_tsv.parent.mkdir(parents=True, exist_ok=True)
    if tmp_dir.exists():
        # mmseqs tmp dirs can be large; reuse path but allow fresh run
        pass
    tmp_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        str(MMSEQS_BIN), "easy-search",
        str(query_db), str(target_db), str(output_tsv), str(tmp_dir),
        "--threads", str(threads),
        "-s", str(EASY_SEARCH_PARAMS["sensitivity"]),
        "--max-seqs", str(EASY_SEARCH_PARAMS["max_seqs"]),
        "-e", str(EASY_SEARCH_PARAMS["evalue"]),
    ]
    print("[CMD]", " ".join(cmd))
    subprocess.run(cmd, check=True)
    if not output_tsv.is_file():
        raise FileNotFoundError(f"结果未生成: {output_tsv}")
    print(f"✅ {output_tsv}")
    return output_tsv


def search_one(method_key: str, skip_existing: bool = True, threads: int | None = None) -> Path:
    ensure_work_dirs()
    engine = method_engine(method_key)
    kwargs = dict(
        query_db=db_prefix(method_key),
        output_tsv=aln_tsv(method_key),
        tmp_dir=aln_tmp_dir(method_key),
        skip_existing=skip_existing,
        threads=threads,
    )
    if engine == "mmseqs":
        return mmseqs_easy_search(**kwargs)
    if engine == "foldseek":
        return foldseek_easy_search(**kwargs)
    raise ValueError(f"未知 engine: {engine} ({method_key})")


def search_all(skip_existing: bool = True, threads: int | None = None) -> dict[str, Path]:
    out: dict[str, Path] = {}
    for _label, key, engine, _di in METHODS:
        print(f"\n══ easy-search: {key} ({engine}) ══")
        out[key] = search_one(key, skip_existing=skip_existing, threads=threads)
    return out


if ONLY_METHODS is None:
    aln_paths = search_all(skip_existing=SKIP_EXISTING, threads=THREADS)
else:
    aln_paths = {}
    for key in ONLY_METHODS:
        print(f"\n══ easy-search: {key} ({method_engine(key)}) ══")
        aln_paths[key] = search_one(key, skip_existing=SKIP_EXISTING, threads=THREADS)

for key, path in aln_paths.items():
    nlines = sum(1 for _ in path.open()) if path.is_file() else 0
    print(f"{key:12s} → {path.name}  lines={nlines:,}")


## Phase B — 评估

1. 使用 `work/lable/scop_lookup.tsv`（由 `0.prepare` 生成；缺失时可重建）
2. 写出 `*_fam/sup/fol.tsv`
3. 汇总 `work/metrics/auc_easy.csv`


In [ ]:
def remove_family_number(scop_class: str) -> str:
    return re.sub(r"\.[0-9]+$", "", scop_class)


def resolve_scop_class(qid: str, id2cls: dict[str, str]) -> str | None:
    candidates: list[str] = [qid]
    base_model = re.sub(r"_MODEL_.*", "", qid)
    if base_model != qid:
        candidates.append(base_model)
    for cand in list(candidates):
        if "." in cand:
            base_chain = re.sub(r"_[A-Za-z0-9]+$", "", cand)
            if base_chain != cand:
                candidates.append(base_chain)
    for cand in candidates:
        if cand in id2cls:
            return id2cls[cand]
    return None


def build_scop_lookup(skip_existing: bool = True) -> Path:
    scop_cla = scop_cla_path()
    ensure_work_dirs()
    if skip_existing and SCOP_LOOKUP.is_file() and SCOP_LOOKUP.stat().st_size > 0:
        print(f"⏭️  SCOP lookup 已存在: {SCOP_LOOKUP}")
        return SCOP_LOOKUP

    id2cls: dict[str, str] = {}
    with scop_cla.open() as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split("\t")
            if len(parts) >= 4:
                id2cls[parts[0].strip()] = parts[3].strip()
    print(f"从 dir.cla 读取了 {len(id2cls)} 个 domain")

    all_ids: set[str] = set()
    with AA_FASTA.open() as f:
        for line in f:
            if line.startswith(">"):
                qid = line[1:].strip().split()[0]
                if qid:
                    all_ids.add(qid)
    print(f"FASTA 中共有 {len(all_ids)} 个唯一 ID")

    missed = 0
    with SCOP_LOOKUP.open("w") as out:
        for qid in sorted(all_ids):
            cls = resolve_scop_class(qid, id2cls)
            if cls is None:
                missed += 1
                continue
            out.write(f"{qid}\t{cls}\n")
    print(f"✅ {SCOP_LOOKUP}  匹配={len(all_ids) - missed} 未匹配={missed}")
    return SCOP_LOOKUP


def load_scop_levels() -> pd.DataFrame:
    rows = []
    with SCOP_LOOKUP.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) < 2:
                continue
            fam = parts[1].strip()
            sf = remove_family_number(fam)
            fo = remove_family_number(sf)
            rows.append({"id": parts[0].strip(), "fa": fam, "sf": sf, "fo": fo})
    return pd.DataFrame(rows)


def calc_fp_rates(aln_tsv_path: Path, cla: pd.DataFrame, out_prefix: Path) -> dict[str, Path]:
    print(f"  读取比对: {aln_tsv_path}", flush=True)
    aln = pd.read_csv(aln_tsv_path, sep="\t", header=None, usecols=[0, 1], names=["qid", "tid"], dtype=str)
    print(f"  原始行数: {len(aln):,}", flush=True)
    aln = aln[aln["qid"] != aln["tid"]].copy()
    print(f"  去 self-hit 后: {len(aln):,}", flush=True)

    work = aln.merge(cla, left_on="qid", right_on="id", how="inner")
    work = work.rename(columns={"fo": "qfo", "sf": "qsf", "fa": "qfa"}).drop(columns=["id"])
    work = work.merge(cla, left_on="tid", right_on="id", how="left")
    work = work.rename(columns={"fo": "tfo", "sf": "tsf", "fa": "tfa"}).drop(columns=["id"])

    is_wrong_fold = (work["qfo"] != work["tfo"]).fillna(True)
    work["seen_fp"] = is_wrong_fold.groupby(work["qid"], sort=False).cumsum().gt(0).astype("int8")

    same_fo = work["qfo"] == work["tfo"]
    same_sf = work["qsf"] == work["tsf"]
    same_fa = work["qfa"] == work["tfa"]

    count_fold = (same_fo & ~same_sf).astype("int32")
    count_super = (same_fo & same_sf & ~same_fa).astype("int32")
    count_family = (same_fo & same_sf & same_fa).astype("int32")

    before_fp = 1 - work["seen_fp"]
    work["fold_tp"] = count_fold * before_fp
    work["super_tp"] = count_super * before_fp
    work["family_tp"] = count_family * before_fp
    work["count_fold"] = count_fold
    work["count_super"] = count_super
    work["count_family"] = count_family

    agg = (
        work.groupby("qid", sort=False)
        .agg(
            focnt=("fold_tp", "sum"), fotot=("count_fold", "sum"),
            sfcnt=("super_tp", "sum"), sftot=("count_super", "sum"),
            facnt=("family_tp", "sum"), fatot=("count_family", "sum"),
        )
        .reset_index()
    )
    for tot in ("fotot", "sftot", "fatot"):
        agg[tot] = agg[tot].replace(0, 1)
    agg["fofrac"] = agg["focnt"] / agg["fotot"]
    agg["sfrac"] = agg["sfcnt"] / agg["sftot"]
    agg["fafrac"] = agg["facnt"] / agg["fatot"]

    out_prefix.parent.mkdir(parents=True, exist_ok=True)
    paths = {
        "fol": Path(str(out_prefix) + "_fol.tsv"),
        "sup": Path(str(out_prefix) + "_sup.tsv"),
        "fam": Path(str(out_prefix) + "_fam.tsv"),
    }
    agg[["qid", "focnt", "fotot", "fofrac"]].to_csv(paths["fol"], sep="\t", header=False, index=False)
    agg[["qid", "sfcnt", "sftot", "sfrac"]].to_csv(paths["sup"], sep="\t", header=False, index=False)
    agg[["qid", "facnt", "fatot", "fafrac"]].to_csv(paths["fam"], sep="\t", header=False, index=False)
    return paths


def mean_sensitivity(level_tsv: Path) -> float:
    vals = []
    with level_tsv.open() as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                vals.append(float(parts[3]))
    return sum(vals) / len(vals) if vals else 0.0


def evaluate_all(skip_existing: bool = True) -> pd.DataFrame:
    ensure_work_dirs()
    build_scop_lookup(skip_existing=skip_existing)
    cla = load_scop_levels()
    print(f"SCOP levels: {len(cla)}")

    auc_rows: list[dict] = []
    for level_name, level_key in (("Family", "fam"), ("Superfamily", "sup"), ("Fold", "fol")):
        row: dict[str, float | str] = {"search_mode": "easy", "level": level_name}
        for label, key, _engine, _di in METHODS:
            tsv_path = aln_tsv(key)
            out_prefix = metric_prefix(key)
            fam_path = Path(str(out_prefix) + "_fam.tsv")
            level_path = Path(str(out_prefix) + f"_{level_key}.tsv")

            print(f"\n[easy] {label}")
            if not tsv_path.is_file():
                print(f"  ❌ 缺少比对: {tsv_path}")
                continue

            if not (skip_existing and fam_path.is_file() and fam_path.stat().st_size > 0):
                if level_name == "Family":
                    try:
                        calc_fp_rates(tsv_path, cla, out_prefix)
                        print("  ✅ 写入 fam/sup/fol")
                    except Exception as e:
                        print(f"  ❌ {e}")
                        traceback.print_exc()
                        continue
                    finally:
                        gc.collect()
            else:
                if level_name == "Family":
                    print("  ⏭️  评估已存在")

            if level_path.is_file():
                row[label] = mean_sensitivity(level_path)
                print(f"  {level_name} AUC={row[label]:.4f}")
        auc_rows.append(row)

    df = pd.DataFrame(auc_rows)
    csv_path = METRICS_DIR / "auc_easy.csv"
    df.to_csv(csv_path, index=False)
    print(f"\n✅ AUC CSV: {csv_path}")
    return df


auc_df = evaluate_all(skip_existing=SKIP_EXISTING)
display(auc_df)
print("\nBenchmark 完成。下一步打开 3.plot.ipynb（作图）")


## 清理临时目录

删除项目根 `tmp/` 与 `work/tmp/`（下载/解压/搜索中间文件）。产物在 `work/` 与 `bin/` 中保留。


In [ ]:
cleanup_tmp(also_work_tmp=True)
